# GEOGLOWS river_id seeding

Assigns each USGS gage its nearest GEOGLOWS v2 reach (`LINKNO` = `river_id`)
by snapping gage points to the GEOGLOWS hydrofabric stream lines.
Complements the NWM half of the seed, which comes from Iman's COMID list
already merged into `merged_gages.geojson`.

**Method** (plan §6): reproject to EPSG:5070 (meters), nearest stream line
within a buffer, keep the snap distance as a crude confidence score, skip
the smallest headwater reaches. Expect ~70% correct — the web app exists
to fix the rest.

**Inputs:** `merged_gages.geojson` + GEOGLOWS v2 hydrofabric (public S3, no credentials).
**Outputs:** `seed.csv` 

In [ ]:
import json
import urllib.request
from pathlib import Path

import geopandas as gpd
import pyogrio
import pandas as pd

import resource
# Hard cap THIS kernel at 8 GB: a runaway cell dies with MemoryError instead of
# swap-thrashing the whole WSL VM (which killed it repeatedly on 2026-07-22).
# Lower-only: raising a finite hard limit is forbidden.
_cap = 8 * 1024**3
_soft, _hard = resource.getrlimit(resource.RLIMIT_AS)
_new_soft = _cap if _hard == resource.RLIM_INFINITY else min(_cap, _hard)
resource.setrlimit(resource.RLIMIT_AS, (_new_soft, _hard))

# ---- Config -----------------------------------------------------------
GAGES_PATH = "../tethysapp/hydro_correlation_tool/public/data/merged_gages.geojson"
BUCKET = "https://geoglows-v2.s3.amazonaws.com"
VPU_BOUNDS_URL = f"/vsicurl/{BUCKET}/hydrography-global/vpu-boundaries.gpkg"
HYDRO_DIR = Path("geoglows_hydrography")   # downloaded streams gpkgs land here (gitignored)
HYDRO_DIR.mkdir(exist_ok=True)

CRS_METERS = "EPSG:5070"   # Albers CONUS — buffer/snap distances in real meters
MAX_SNAP_M = 500           # no reach within this distance -> no match (tune empirically)
MIN_STREAM_ORDER = 2       # skip order-1 headwaters so gages don't snap to tiny tributaries

In [ ]:
# ---- Load gages -------------------------------------------------------
gages = gpd.read_file(GAGES_PATH)
print(len(gages), "gages |", gages.crs)
gages.head(3)

## 1. Which VPUs cover CONUS?

The hydrofabric is split into 125 VPU (Vector Processing Unit) regions.
`/vsicurl/` + a bbox reads only the CONUS window of the 1.9 GB boundaries
file over HTTP instead of downloading it.

In [ ]:
conus_bbox = tuple(gages.total_bounds)  # (minx, miny, maxx, maxy) in 4326
vpus = gpd.read_file(VPU_BOUNDS_URL, bbox=conus_bbox)
print(vpus.columns.tolist())
vpus.head(10)

In [ ]:
print(vpus['VPU'])

In [ ]:
# Auto-detect the VPU-code column (name not documented; inspect the print above
# and hard-code it here if this guess picks the wrong one).
vpu_col = next(c for c in vpus.columns if "vpu" in c.lower())
print("Using VPU column:", vpu_col)
vpus[vpu_col] = pd.to_numeric(vpus[vpu_col]).astype("Int64")  # codes ship as strings

# A gage belongs to the VPU polygon that contains it.
gage_vpu = gpd.sjoin(gages, vpus[[vpu_col, "geometry"]].to_crs(gages.crs),
                     how="left", predicate="within")

# Coastal/border gages can fall just outside every polygon -> nearest VPU
missing = gage_vpu[vpu_col].isna()
if missing.any():
    near = gpd.sjoin_nearest(gages.loc[missing.values, ["USGSID", "geometry"]].to_crs(CRS_METERS),
                             vpus[[vpu_col, "geometry"]].to_crs(CRS_METERS), how="left")
    gage_vpu.loc[missing, vpu_col] = near[vpu_col].values
    print(f"assigned {missing.sum()} stray gages to nearest VPU")

conus_vpus = sorted(gage_vpu[vpu_col].dropna().unique())
print("CONUS VPUs:", conus_vpus)
print("Gages per VPU:")
print(gage_vpu[vpu_col].value_counts(dropna=False))

## 2. Download the streams file for each CONUS VPU

~270 MB per VPU; skips files already downloaded, so re-running is cheap.

In [ ]:
def streams_path(vpu):
    return HYDRO_DIR / f"streams_{vpu}.gpkg"

for vpu in conus_vpus:
    dest = streams_path(vpu)
    if dest.exists():
        print(f"vpu={vpu}: already downloaded")
        continue
    url = f"{BUCKET}/hydrography/vpu={vpu}/streams_{vpu}.gpkg"
    print(f"vpu={vpu}: downloading {url} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"  -> {dest} ({dest.stat().st_size / 1e6:.0f} MB)")

## 3. Snap gages to their nearest reach, one VPU at a time

`sjoin_nearest` uses a spatial index under the hood; `distance_col` records
how far each gage had to snap — small = trustworthy, large = review first.

In [ ]:
matches = []
id_col = order_col = None

for vpu in conus_vpus:
    sub = gage_vpu[gage_vpu[vpu_col] == vpu][["USGSID", "geometry"]]
    if sub.empty:
        continue

    # bbox-filtered read keeps memory sane. NOTE: the bbox must be in the
    # FILE's CRS — the hydrofabric ships in EPSG:3857 (meters), not 4326,
    # so transform the gage bounds first or the filter selects nothing.
    file_crs = pyogrio.read_info(streams_path(vpu))["crs"]
    minx, miny, maxx, maxy = sub.to_crs(file_crs).total_bounds
    margin = 5000  # meters, ~5 km
    streams = gpd.read_file(streams_path(vpu),
                            bbox=(minx - margin, miny - margin, maxx + margin, maxy + margin))

    if id_col is None:  # detect field names once, from real data
        print("Stream columns:", streams.columns.tolist())
        id_col = next(c for c in streams.columns if c.lower() == "linkno")
        order_col = next((c for c in streams.columns if "order" in c.lower()), None)
        print(f"Using id column: {id_col!r}, stream-order column: {order_col!r}")

    if order_col is not None:
        streams = streams[streams[order_col] >= MIN_STREAM_ORDER]

    m = gpd.sjoin_nearest(
        sub.to_crs(CRS_METERS),
        streams[[id_col, "geometry"]].to_crs(CRS_METERS),
        how="left", max_distance=MAX_SNAP_M, distance_col="geoglows_snap_m",
    )
    matched = m[id_col].notna().sum()
    print(f"vpu={vpu}: {matched}/{len(sub)} gages matched")
    matches.append(m[["USGSID", id_col, "geoglows_snap_m"]])

In [ ]:
# Combine; a gage can appear twice if two reaches tie for nearest -> keep the closest
result = (pd.concat(matches, ignore_index=True)
            .sort_values("geoglows_snap_m")
            .drop_duplicates("USGSID", keep="first")
            .rename(columns={id_col: "geoglows_river_id"}))
result["geoglows_river_id"] = result["geoglows_river_id"].astype("Int64")

seed = gages.merge(result, on="USGSID", how="left")
print(f"{seed['geoglows_river_id'].notna().sum()}/{len(seed)} gages got a geoglows_river_id")

## 4. QA — how far did gages have to snap?

Big distances are the matches to distrust (and exactly what the app's
verify workflow is for).

In [ ]:
print(seed["geoglows_snap_m"].describe())
seed["geoglows_snap_m"].plot.hist(bins=50, title="Snap distance (m)")

In [ ]:
# The 20 most-suspicious matches — worth eyeballing in the app later
seed.nlargest(20, "geoglows_snap_m")[["USGSID", "station_nm", "geoglows_river_id", "geoglows_snap_m"]]

## 5. Save the seed

`seed.csv` is the preprocessing input

gage_mappig.csv is now used as the database initializer, as it holds the current (8/18/2026) verification table

In [ ]:

csv_out = pd.DataFrame({
    "usgs_id": seed["USGSID"],
    "gage_name": seed["station_nm"],
    "latitude": seed.geometry.y,
    "longitude": seed.geometry.x,
    "nwm_feature_id": seed["COMID"],
    "geoglows_river_id": seed["geoglows_river_id"],
    "verification_status": "Unverified",
})
csv_out.to_csv("seed.csv", index=False)
print("Wrote seed.csv")
csv_out.head()